# StrikeVision — Pose Estimation

Pipeline general para cualquier round: `PersonDetector@production` + ByteTrack + MediaPipe Pose Landmarker. La salida conserva los IDs de cada fragmento de track y genera keypoints en coordenadas del frame original.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml

from ufc_tracker.detection.weights import project_root
from ufc_tracker.pose.pipeline import run_pose_pipeline

ROOT = project_root(Path.cwd())
CONFIG_PATH = ROOT / 'configs/app/pose_pipeline.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config

## Seleccionar un round

La tabla usa el manifiesto versionado del dataset. Cambia `VIDEO_RELATIVE_PATH` por cualquier fila antes de ejecutar el pipeline.

In [ ]:
manifest = pd.read_csv(ROOT / 'data/metadata/splits_manifest.csv')
display(manifest[['category', 'fight_id', 'round_number', 'relative_path']])
VIDEO_RELATIVE_PATH = manifest.iloc[0]['relative_path']
VIDEO_PATH = ROOT / VIDEO_RELATIVE_PATH
OUTPUT_DIR = ROOT / config['output_root'] / VIDEO_PATH.stem
VIDEO_PATH, OUTPUT_DIR

## Ejecutar MediaPipe Pose

Usa `max_frames=300` para desarrollo. Cambia a `None` únicamente después de revisar el preview corto.

In [ ]:
result = run_pose_pipeline(
    VIDEO_PATH,
    OUTPUT_DIR,
    tracking_confidence=float(config['tracking_confidence']),
    min_track_frames=int(config['min_track_frames']),
    max_frames=300,
)
result

In [ ]:
metrics = json.loads(result.metrics_path.read_text(encoding='utf-8'))
metrics['metrics']['overall']

## Criterios de revisión

Abre `result.preview_path` y revisa muñecas, codos, rodillas y tobillos durante guardia, ataque y oclusiones. `pose_coverage` mide poses con al menos cuatro keypoints esenciales; la disponibilidad por keypoint muestra qué articulaciones se pierden.